# Avanza tu proyecto

**Sesión 8 · IA Intermedio · EstudIA 2026**

Hoy avanzas tu proyecto con lo que vimos: buscar por significado, agrupar sin etiquetas y describir clases con
palabras. **Te recomendamos usar tus propios datos**: cuantos más ejemplos tuyos etiquetes, mejor mides, y según la
curva de la sesión 7, más cerca estás de que el ajuste fino valga la pena. Si todavía no tienes datos, usa uno de los
caminos de práctica.

| Camino | Para quién | Qué haces |
|---|---|---|
| **A1. Tu proyecto de texto** | Tu proyecto clasifica textos | Buscar en tus textos por significado, agrupar y comparar con tus etiquetas, mejorar las descripciones de tus etiquetas |
| **A2. Tu proyecto de imágenes** | Tu proyecto clasifica imágenes | Zero-shot con descripciones, tus fotos etiquetadas como ejemplos, buscar con palabras y agrupar |
| **B1. Práctica con imágenes** | Todavía no tienes fotos | A2 con las hojas de café de la sesión 4 |
| **B2. Práctica con texto** | Todavía no tienes textos | A1 con los 30 mensajes ciudadanos |

Cada resultado queda en tu **registro de experimentos**. Cuando termines, **entrega el notebook** desde la última
celda. No importa si el modelo acierta.

**Antes de empezar.** *Archivo › Guardar una copia en Drive*. No necesitas GPU.

In [ ]:
# Preparación: librerías. Ejecútala una vez.
import os, re, time, json, zipfile, urllib.request, warnings
from collections import Counter
warnings.filterwarnings("ignore")
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s08")
ARCHIVOS_S04 = ARCHIVOS.replace("/s08", "/s04")
MODELO = "google/siglip2-base-patch16-224"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display
torch.set_num_threads(max(1, os.cpu_count() or 1))

try:
    from sentence_transformers import SentenceTransformer
except ImportError:  # Colab ya la trae; fuera de Colab puede faltar
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
    from sentence_transformers import SentenceTransformer

print("Listo.", "Con GPU." if torch.cuda.is_available() else "Sin GPU: alcanza para todo el notebook.")

In [ ]:
# @title Funciones auxiliares: números, evaluar, vectores de fotos y textos, agrupar y registrar
def descargar(nombre, url=None):
    """Baja un archivo (de la carpeta pública del curso, o de `url`) solo la primera vez y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", os.path.basename(nombre))
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(url or f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

def tokens(texto):
    """Parte un texto en palabras y signos, en minúsculas: "No me gustó." -> ["no", "me", "gustó", "."]."""
    return re.findall(r"\w+|[^\w\s]", str(texto).lower())

def evaluar(verdad, prediccion, titulo="Tu modelo"):
    """Compara las predicciones con las etiquetas reales y con la línea base de la clase más frecuente."""
    verdad, prediccion = [str(v) for v in verdad], [str(p) for p in prediccion]
    clases = sorted(set(verdad))
    frecuente = max(clases, key=verdad.count)
    def balanceada(p):
        return float(np.mean([np.mean([pi == c for vi, pi in zip(verdad, p) if vi == c]) for c in clases]))
    print(f"{titulo} · {len(verdad)} ejemplos, {len(clases)} clases")
    print(f"  Exactitud:            {pct(np.mean([v == p for v, p in zip(verdad, prediccion)]))}   "
          f"(línea base “siempre {frecuente}”: {pct(np.mean([v == frecuente for v in verdad]))})")
    print(f"  Exactitud balanceada: {pct(balanceada(prediccion))}   (línea base: {pct(balanceada([frecuente] * len(verdad)))})")
    todas = clases + sorted(set(prediccion) - set(clases))
    fig, eje = plt.subplots(figsize=(1.1 * len(todas) + 2.5, 0.9 * len(clases) + 1.8))
    ConfusionMatrixDisplay(confusion_matrix(verdad, prediccion, labels=todas), display_labels=todas).plot(
        ax=eje, cmap="Purples", colorbar=False, xticks_rotation=45)
    eje.set_xlabel("Lo que respondió"); eje.set_ylabel("La etiqueta real"); eje.set_title(titulo)
    eje.set_ylim(len(clases) - 0.5, -0.5)
    plt.tight_layout(); plt.show()

def _pliegues(etiquetas):
    n = int(min(5, np.bincount(np.unique(etiquetas, return_inverse=True)[1]).min()))
    if n < 2:
        print("Necesitas al menos 2 ejemplos de cada etiqueta.")
    return n


def bolsa_con_validacion_cruzada(textos, etiquetas):
    """Bolsa de palabras + regresión logística, medida con validación cruzada (la de la sesión 5)."""
    etiquetas = np.array([str(e) for e in etiquetas])
    n = _pliegues(etiquetas)
    if n < 2:
        return None
    modelo = make_pipeline(TfidfVectorizer(tokenizer=tokens, lowercase=False, token_pattern=None, ngram_range=(1, 2)),
                           LogisticRegression(max_iter=3000, C=10, class_weight="balanced"))
    return cross_val_predict(modelo, list(textos), etiquetas, cv=StratifiedKFold(n, shuffle=True, random_state=0))

def embeddings_con_validacion_cruzada(X, etiquetas, C=10):
    """Regresión logística sobre embeddings ya calculados (una fila por ejemplo), con validación cruzada."""
    etiquetas = np.array([str(e) for e in etiquetas])
    n = _pliegues(etiquetas)
    if n < 2:
        return None
    return cross_val_predict(LogisticRegression(max_iter=5000, C=C, class_weight="balanced"), X, etiquetas,
                             cv=StratifiedKFold(n, shuffle=True, random_state=0))

def zero_shot(X, etiquetas_posibles, D):
    """A cada fila de X le da la etiqueta cuya descripción (fila de D) queda más cerca."""
    return [etiquetas_posibles[i] for i in np.argmax(X @ D.T, axis=1)]

def parecidos_con_otra_etiqueta(X, ejemplos, etiquetas, cuantos=5):
    """Los pares de ejemplos más parecidos entre sí que tienen etiquetas distintas: posibles etiquetas dudosas."""
    S = X @ X.T
    pares = [(S[i, j], i, j) for i in range(len(X)) for j in range(i + 1, len(X)) if etiquetas[i] != etiquetas[j]]
    for s, i, j in sorted(pares, reverse=True)[:cuantos]:
        print(f"{s:.2f}".replace(".", ","), f"· [{etiquetas[i]}] {str(ejemplos[i])[:70]}")
        print("     ", f"[{etiquetas[j]}] {str(ejemplos[j])[:70]}")

def exactitudes(verdad, prediccion):
    """Exactitud y exactitud balanceada, en porcentaje, para el registro."""
    verdad, prediccion = np.array([str(v) for v in verdad]), np.array([str(p) for p in prediccion])
    bal = np.mean([np.mean(prediccion[verdad == c] == c) for c in np.unique(verdad)])
    return round(100 * float(np.mean(verdad == prediccion)), 1), round(100 * float(bal), 1)

def prototipos_con_validacion_cruzada(X, etiquetas):
    """Cada foto se clasifica con el promedio de los vectores de las fotos etiquetadas de cada clase, sin usarla a
    ella misma: “aprende” solo con tus fotos, sin entrenar nada."""
    etiquetas = np.array([str(e) for e in etiquetas])
    clases = sorted(set(etiquetas))
    pred = []
    for i in range(len(X)):
        otras = np.arange(len(X)) != i
        P = np.stack([X[otras & (etiquetas == c)].mean(0) for c in clases])
        pred.append(clases[int(np.argmax(P @ X[i]))])
    return pred

def grupos_frente_a_etiquetas(X, etiquetas, k=None):
    """Agrupa con k-medias (sin mirar las etiquetas) y muestra cómo se reparten tus etiquetas en los grupos."""
    from sklearn.cluster import KMeans
    k = k or len(set(etiquetas))
    grupos = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    display(pd.crosstab(pd.Series(grupos + 1, name="grupo"), pd.Series(etiquetas, name="tu etiqueta")))
    return grupos

from transformers import AutoModel, AutoProcessor
from transformers.utils import logging as registro_hf
registro_hf.set_verbosity_error()
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
_cargados = {}

def _modelo(nombre):
    if nombre not in _cargados:
        _cargados[nombre] = (AutoModel.from_pretrained(nombre).to(DISPOSITIVO).eval(), AutoProcessor.from_pretrained(nombre))
    return _cargados[nombre]

def _unitario(v):
    v = getattr(v, "pooler_output", v)           # según la versión, viene suelto o dentro de un objeto
    return torch.nn.functional.normalize(v, dim=-1).float().cpu().numpy()

def vectores_de_fotos(fotos, modelo=None, lote=32):
    """Un vector de largo 1 por foto (fotos de PIL o rutas a archivos)."""
    red, proc = _modelo(modelo or MODELO)
    salida = []
    with torch.no_grad():
        for i in range(0, len(fotos), lote):
            grupo = [f if isinstance(f, Image.Image) else Image.open(f) for f in fotos[i:i + lote]]
            grupo = [f.convert("RGB") for f in grupo]
            salida.append(_unitario(red.get_image_features(**proc(images=grupo, return_tensors="pt").to(DISPOSITIVO))))
    return np.concatenate(salida)

def vectores_de_textos(textos, modelo=None):
    """Un vector de largo 1 por texto, en el mismo espacio que las fotos."""
    nombre = modelo or MODELO
    red, proc = _modelo(nombre)
    relleno = {"padding": "max_length", "max_length": 64} if "siglip" in nombre else {"padding": True}
    with torch.no_grad():
        entrada = proc(text=list(textos), return_tensors="pt", truncation=True, **relleno).to(DISPOSITIVO)
        return _unitario(red.get_text_features(**entrada))

def mostrar(fotos, titulos=None, columnas=6, tamano=2.0, colores=None):
    """Una cuadrícula de fotos con su título."""
    filas = (len(fotos) + columnas - 1) // columnas
    fig, ejes = plt.subplots(filas, columnas, figsize=(tamano * columnas, tamano * filas + 0.3 * filas))
    for k, eje in enumerate(np.atleast_1d(ejes).ravel()):
        eje.axis("off")
        if k < len(fotos):
            f = fotos[k] if isinstance(fotos[k], Image.Image) else Image.open(fotos[k])
            eje.imshow(f.convert("RGB"))
            if titulos:
                eje.set_title(titulos[k], fontsize=9, color=(colores[k] if colores else "black"))
    plt.tight_layout(); plt.show()

def abrir_zip(nombre):
    """Descarga (la primera vez) y descomprime un zip de la carpeta del curso; devuelve sus fotos ordenadas."""
    ruta = descargar(nombre)
    with zipfile.ZipFile(ruta) as z:
        z.extractall("archivos_curso")
        return sorted(os.path.join("archivos_curso", n) for n in z.namelist() if n.lower().endswith(".jpg"))

from IPython.display import display
REGISTRO = "experimentos.csv"

def registrar(**datos):
    """Agrega una fila a tu registro de experimentos (el archivo experimentos.csv): qué probaste y qué salió."""
    fila = pd.DataFrame([{"hora": time.strftime("%H:%M"), **datos}])
    tabla = pd.concat([pd.read_csv(REGISTRO), fila], ignore_index=True) if os.path.exists(REGISTRO) else fila
    tabla.to_csv(REGISTRO, index=False)

def ver_registro():
    """Muestra el registro de experimentos, el más reciente abajo."""
    if not os.path.exists(REGISTRO):
        print("Todavía no hay experimentos en el registro.")
        return
    tabla = pd.read_csv(REGISTRO)
    display(tabla.style.format(precision=1, decimal=",", thousands=".", na_rep="").hide(axis="index"))

In [ ]:
_e5 = SentenceTransformer("intfloat/multilingual-e5-small", device="cuda" if torch.cuda.is_available() else "cpu")

def embeber(textos):
    """Vectores de textos con e5 (sesión 6), en minúsculas y con el prefijo que pide el modelo."""
    return _e5.encode(["query: " + str(t).lower() for t in textos], batch_size=32, normalize_embeddings=True,
                      show_progress_bar=False)

def camino_texto(camino, textos, etiquetas, descripciones, busquedas):
    """Buscar, agrupar y describir con tus textos."""
    X = embeber(textos)
    print("1. Buscar por significado")
    for q in busquedas:
        s = X @ embeber([q])[0]
        print(f"   “{q}”")
        for i in np.argsort(-s)[:3]:
            print(f"      {s[i]:.2f}  [{etiquetas[i]}] {str(textos[i])[:80]}".replace(".", ",", 1))
    print("\n2. Grupos sin etiquetas frente a tus etiquetas")
    grupos_frente_a_etiquetas(X, etiquetas)
    print("3. Zero-shot con tus descripciones")
    clases = sorted(set(etiquetas))
    D = embeber([descripciones.get(c, c.replace("_", " ")) for c in clases])
    pred = zero_shot(X, clases, D)
    evaluar(etiquetas, pred, "Zero-shot con tus descripciones")
    e, b = exactitudes(etiquetas, pred)
    registrar(camino=camino, modelo="e5 pequeño", forma="zero-shot con descripciones", ejemplos=len(textos),
              exactitud=e, exactitud_balanceada=b)

def camino_imagen(camino, fotos, etiquetas, descripciones, busquedas):
    """Zero-shot con descripciones, tus fotos como ejemplos, buscar con palabras y agrupar."""
    clases = sorted(set(etiquetas))
    inicio = time.time()
    X = vectores_de_fotos(fotos)
    print(f"{len(fotos)} fotos en vectores en {time.time() - inicio:.0f} s.")
    T = vectores_de_textos([descripciones.get(c, c.replace("_", " ")) for c in clases])
    pred = np.array(clases)[(X @ T.T).argmax(1)]
    evaluar(etiquetas, pred, "1. Zero-shot con tus descripciones")
    e, b = exactitudes(etiquetas, pred)
    registrar(camino=camino, modelo=MODELO, forma="zero-shot con descripciones", ejemplos=len(fotos),
              exactitud=e, exactitud_balanceada=b)
    pred = prototipos_con_validacion_cruzada(X, etiquetas)
    evaluar(etiquetas, pred, "2. Tus fotos etiquetadas como ejemplos (validación cruzada)")
    e, b = exactitudes(etiquetas, pred)
    registrar(camino=camino, modelo=MODELO, forma="promedio de tus fotos etiquetadas", ejemplos=len(fotos),
              exactitud=e, exactitud_balanceada=b)
    print("3. Buscar con palabras")
    for q in busquedas:
        s = X @ vectores_de_textos([q])[0]
        mejores = np.argsort(-s)[:5]
        print(f"   “{q}”")
        mostrar([fotos[i] for i in mejores], [f"{etiquetas[i]} · {s[i]:.2f}".replace(".", ",") for i in mejores],
                columnas=5, tamano=2)
    print("4. Grupos sin etiquetas frente a tus etiquetas")
    grupos_frente_a_etiquetas(X, etiquetas)

---
## A1. Tu proyecto de texto

Tus ejemplos (los de las sesiones anteriores, **y los nuevos que etiquetes**) en `EJEMPLOS` o en `ejemplos.csv`
(columnas `entrada` y `etiqueta`). Sin datos personales. En `DESCRIPCIONES`, una frase por etiqueta; en `BUSQUEDAS`,
frases para buscar entre tus textos.

In [ ]:
EJEMPLOS = [
    ("Escribe aquí un ejemplo real de tu proyecto.", "etiqueta_1"),
    ("Y otro, con otra etiqueta.", "etiqueta_2"),
]
DESCRIPCIONES = {"etiqueta_1": "una frase que describe la etiqueta 1", "etiqueta_2": "una frase que describe la etiqueta 2"}
BUSQUEDAS = ["una frase para buscar entre tus textos"]

if os.path.exists("ejemplos.csv"):
    tabla = pd.read_csv("ejemplos.csv")
    EJEMPLOS = list(zip(tabla["entrada"].astype(str), tabla["etiqueta"].astype(str)))
if not EJEMPLOS[0][0].startswith("Escribe aquí"):
    textos, etiquetas = [e for e, _ in EJEMPLOS], [str(et) for _, et in EJEMPLOS]
    print(f"{len(textos)} ejemplos:", Counter(etiquetas))
    camino_texto("A1", textos, etiquetas, DESCRIPCIONES, BUSQUEDAS)
else:
    print("Camino A1: escribe tus ejemplos en EJEMPLOS (o sube ejemplos.csv) y vuelve a ejecutar.")

---
## A2. Tu proyecto de imágenes

Sube `mis_fotos.zip` con una carpeta por etiqueta (por ejemplo, `mis_fotos/sana/…jpg`), sin personas reconocibles. En
`DESCRIPCIONES_FOTOS`, una frase por etiqueta (en español, o en inglés si te va mejor); en `BUSQUEDAS_FOTOS`, frases
para buscar entre tus fotos.

In [ ]:
DESCRIPCIONES_FOTOS = {"etiqueta_1": "una foto de …", "etiqueta_2": "una foto de …"}
BUSQUEDAS_FOTOS = ["una foto de …"]

if os.path.exists("mis_fotos.zip"):
    with zipfile.ZipFile("mis_fotos.zip") as z:
        z.extractall(".")
    rutas = sorted(os.path.join(r, f) for r, _, fs in os.walk("mis_fotos") for f in fs
                   if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")))
    etiquetas_img = [os.path.basename(os.path.dirname(r)) for r in rutas]
    print(f"{len(rutas)} fotos:", Counter(etiquetas_img))
    camino_imagen("A2", rutas, etiquetas_img, DESCRIPCIONES_FOTOS, BUSQUEDAS_FOTOS)
else:
    print("Camino A2: sube mis_fotos.zip y vuelve a ejecutar.")

---
## B1. Práctica con imágenes: las hojas de café

Las 320 hojas de café etiquetadas de la sesión 4 (sana, ácaro rojo, roya leve, roya avanzada). Cambia las
descripciones y vuelve a ejecutar.

In [ ]:
CAFE = {'sano': 'una foto de una hoja de café sana', 'acaro_rojo': 'una foto de una hoja de café con ácaro rojo', 'roya_leve': 'una foto de una hoja de café con pocas manchas de roya', 'roya_avanzada': 'una foto de una hoja de café con mucha roya anaranjada'}
BUSQUEDAS_CAFE = ["una hoja con manchas anaranjadas", "una hoja verde y brillante"]
EJECUTAR_B1 = True

if EJECUTAR_B1:
    ruta = descargar("cafe_entrenamiento.zip", f"{ARCHIVOS_S04}/cafe_entrenamiento.zip")
    with zipfile.ZipFile(ruta) as z:
        z.extractall("archivos_curso")
        fotos_cafe = sorted(os.path.join("archivos_curso", n) for n in z.namelist() if n.endswith(".jpg"))
    etiquetas_cafe = [f.replace(os.sep, "/").split("/")[-2] for f in fotos_cafe]
    camino_imagen("B1", fotos_cafe, etiquetas_cafe, CAFE, BUSQUEDAS_CAFE)

---
## B2. Práctica con texto: los mensajes ciudadanos

Los 30 mensajes de las sesiones anteriores, cinco por tema.

In [ ]:
MENSAJES = [
    ("El semáforo de la 80 con la 33 lleva tres días apagado y se forman unos trancones terribles.", "movilidad"),
    ("El metro iba tan lleno esta mañana que no pude subirme en tres trenes.", "movilidad"),
    ("Los buses de la ruta del barrio dejaron de pasar después de las ocho de la noche.", "movilidad"),
    ("Hay un hueco enorme en la vía principal y ya se han caído dos motociclistas.", "movilidad"),
    ("No hay señalización en la nueva ciclorruta y los carros se parquean encima.", "movilidad"),
    ("Anoche robaron dos celulares en el paradero del parque y no había ningún policía.", "seguridad"),
    ("Pusieron cámaras en la ciclorruta, pero siguen robando bicicletas.", "seguridad"),
    ("En la esquina de la cancha venden droga a plena luz del día.", "seguridad"),
    ("Las luces del puente peatonal están dañadas y da miedo pasar de noche.", "seguridad"),
    ("Unos muchachos están cobrando vacuna a los tenderos de la cuadra.", "seguridad"),
    ("En el centro de salud del barrio no hay citas para medicina general hasta el próximo mes.", "salud"),
    ("Llevo tres horas esperando en urgencias con mi mamá y nadie nos atiende.", "salud"),
    ("La farmacia de la EPS no tiene la insulina desde hace dos semanas.", "salud"),
    ("Hay muchos casos de dengue en el sector y nadie ha venido a fumigar.", "salud"),
    ("Necesitamos una jornada de vacunación para los niños de la vereda.", "salud"),
    ("La escuela de la vereda no tiene profesor de matemáticas desde marzo.", "educación"),
    ("Al colegio no ha llegado el refrigerio de los estudiantes en todo el mes.", "educación"),
    ("Los computadores de la sala de informática del colegio llevan un año dañados.", "educación"),
    ("No hay cupos en el preescolar del barrio y tengo que trabajar.", "educación"),
    ("El techo del salón se cae a pedazos cuando llueve y los niños reciben clase así.", "educación"),
    ("Hace una semana se fue el agua en toda la cuadra y nadie da razón.", "servicios públicos"),
    ("La factura de la luz llegó el doble de lo normal sin ninguna explicación.", "servicios públicos"),
    ("Hay una fuga de gas en el contador del edificio y huele muy fuerte.", "servicios públicos"),
    ("El alcantarillado se rebosa cada vez que llueve y el agua entra a las casas.", "servicios públicos"),
    ("Se cayó un cable de energía en el andén y está echando chispas.", "servicios públicos"),
    ("Están quemando basura al lado de la quebrada y el humo llega a todas las casas.", "medio ambiente"),
    ("Talaron los árboles del parque sin ningún permiso.", "medio ambiente"),
    ("Una fábrica está botando aguas negras al río.", "medio ambiente"),
    ("La calidad del aire está pésima y los niños tienen tos todo el día.", "medio ambiente"),
    ("Dejan escombros y colchones viejos en la orilla de la quebrada.", "medio ambiente"),
]
DESCRIPCIONES_B2 = {"movilidad": "tránsito y transporte", "seguridad": "seguridad y delincuencia",
                    "salud": "salud y hospitales", "educación": "educación y colegios",
                    "servicios públicos": "agua, luz o gas", "medio ambiente": "medio ambiente y contaminación"}
BUSQUEDAS_B2 = ["se fue la luz en el barrio", "robaron un celular"]
EJECUTAR_B2 = True

if EJECUTAR_B2:
    camino_texto("B2", [m for m, _ in MENSAJES], [t for _, t in MENSAJES],
                 {t: f"Este mensaje trata sobre {d}." for t, d in DESCRIPCIONES_B2.items()}, BUSQUEDAS_B2)

---
## Tu registro y lo que avanzaste

In [ ]:
ver_registro()

**Lo que avancé** (doble clic aquí y responde en dos o tres líneas por pregunta):

- ¿Cuántos ejemplos tienes hoy en tu proyecto? ¿Cuántos más etiquetaste esta semana?
- ¿Qué encontraste al agrupar? ¿Algún grupo mezcla etiquetas que deberían estar separadas, o separa una etiqueta en
  dos? ¿Cambiarías la definición de alguna etiqueta?
- ¿Qué descripción de etiqueta te funcionó mejor y por qué?
- ¿Cuál es el siguiente paso de tu proyecto?

---
## Entrega

Ejecuta las dos celdas siguientes. La primera solo prepara la función; la segunda lee este notebook con sus
resultados y lo envía a la plataforma. Te pide tu clave la primera vez.

Si algo falla, descarga el notebook (*Archivo › Descargar › Descargar .ipynb*) y súbelo en la plataforma, en la
actividad “Entrega: avanza tu proyecto”.

In [ ]:
# @title Función para entregar el notebook (no hace falta cambiarla)
import getpass, json, urllib.error

PLATAFORMA = os.environ.get("ESTUDIA_PLATAFORMA", "https://estud-ia.eslopezag.com")
_CLAVE = os.environ.get("ESTUDIA_CLAVE", "")
ACTIVIDAD = "“Entrega: avanza tu proyecto”"

def _este_notebook():
    """El notebook tal como se ve en Colab, con sus resultados. Fuera de Colab, el archivo de ESTUDIA_NOTEBOOK."""
    if os.environ.get("ESTUDIA_NOTEBOOK"):
        with open(os.environ["ESTUDIA_NOTEBOOK"], encoding="utf-8") as f:
            return json.load(f)
    try:
        from google.colab import _message
        return _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
    except Exception:
        return None

def entregar():
    """Envía este notebook, con lo que ejecutaste, a la plataforma del curso."""
    global _CLAVE
    cuaderno = _este_notebook()
    if not cuaderno:
        print("No pude leer el notebook desde aquí. Descárgalo (Archivo › Descargar › Descargar .ipynb) y súbelo "
              f"en la plataforma, en la actividad {ACTIVIDAD}.")
        return
    if not _CLAVE:
        _CLAVE = getpass.getpass(f"Pega tu clave (plataforma › {ACTIVIDAD} › Copiar) y oprime Enter: ").strip()
    req = urllib.request.Request(f"{PLATAFORMA}/api/entregas/s08", data=json.dumps({"notebook": cuaderno}).encode(),
                                 method="POST", headers={"Content-Type": "application/json",
                                                         "Authorization": f"Bearer {_CLAVE}",
                                                         "User-Agent": "EstudIA-notebook/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            res = json.loads(r.read())
    except urllib.error.HTTPError as e:
        res = json.loads(e.read() or b"{}")
        if e.code == 401:
            _CLAVE = ""
        print("No se pudo entregar:", res.get("error", f"error {e.code}"))
        print(f"Si no se arregla, descarga el notebook y súbelo en la plataforma, en la actividad {ACTIVIDAD}.")
        return
    except urllib.error.URLError:
        print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")
        return
    modelos = ", ".join(res["modelos"]) or "ninguno"
    print(f"Entrega recibida. Celdas de código ejecutadas: {res['ejecutadas']} de {res['celdas']}. "
          f"Modelos de Hugging Face que encontré: {modelos}.")
    for aviso in res.get("avisos", []):
        print("Ojo:", aviso)

In [ ]:
entregar()